1. Build per-attack annotation CSVs

In [ ]:
from pathlib import Path
import pandas as pd
import os

CSV_PATH = Path(r"C:\Users\evri\Desktop\Διπλωματικη\Adversarial AI Attack Detection\Adversarial-AI-Attack-Detection\fruit_quality_split_annotations.csv")
ADV_ROOT  = Path(r"E:\adv_outputs_fruit_quality")  
OUT_DIR   = Path(r"C:\Users\evri\Desktop\Διπλωματικη\Adversarial AI Attack Detection\Adversarial-AI-Attack-Detection\fruit_quality_adv_annotations")

OUT_DIR.mkdir(parents=True, exist_ok=True)

MAKE_RELATIVE = False
REL_PREFIX = Path("adv_outputs")  

label_map = {0: "good", 1: "bad", 2: "mixed"}

def load_csv_norm(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["image_id_norm"] = df["image_id"].astype(str).str.replace("\\\\", "/", regex=True).str.strip()
    df["image_id_norm_lc"] = df["image_id_norm"].str.lower()
    df["basename"] = df["image_id_norm"].apply(lambda s: Path(s).name)
    return df

df = load_csv_norm(CSV_PATH)
print(f"Loaded CSV: {len(df)} rows. Example: {df[['image_id','split','label']].head(2).to_dict(orient='records')}")

if not ADV_ROOT.exists():
    raise FileNotFoundError(f"ADV_ROOT not found: {ADV_ROOT}")
attack_dirs = sorted([p for p in ADV_ROOT.iterdir() if p.is_dir()])
if not attack_dirs:
    raise FileNotFoundError(f"No attack subfolders in ADV_ROOT: {ADV_ROOT}")
print("Attacks discovered:", [p.name for p in attack_dirs])

EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

def find_adv_path(attack_root: Path, attack_name: str, split: str, label_name: str, basename: str) -> Path:
    base = Path(basename)
    cand0 = attack_root / attack_name / split / label_name / base.name
    if cand0.exists():
        return cand0
    stem = base.stem
    for ext in EXTS:
        cand = attack_root / attack_name / split / label_name / (stem + ext)
        if cand.exists():
            return cand
    return None

attack_file_index = {}
for ad in attack_dirs:
    attack_name = ad.name
    attack_file_index[attack_name] = set()
    for split_dir in ad.iterdir():
        if not split_dir.is_dir(): 
            continue
        for label_dir in split_dir.iterdir():
            if not label_dir.is_dir():
                continue
            for f in label_dir.iterdir():
                attack_file_index[attack_name].add(str(f.name).lower())  

summary = {}
for ad in attack_dirs:
    attack_name = ad.name
    print(f"\nProcessing attack: {attack_name}")
    rows = []
    missing = 0
    found = 0
    for _, row in df.iterrows():
        orig_path = row["image_id_norm"]        
        key_full = str(orig_path).replace("\\", "/")
        basename = row["basename"]
        split = str(row["split"]).strip()
        lab = int(row["label"])
        label_name = label_map.get(lab, f"class_{lab}")

        adv_candidate = find_adv_path(ADV_ROOT, attack_name, split, label_name, basename)
        if adv_candidate is None:
            if basename.lower() in attack_file_index.get(attack_name, set()):
                found_path = None
                for split_dir in (ADV_ROOT / attack_name).iterdir():
                    if not split_dir.is_dir(): continue
                    for label_dir in split_dir.iterdir():
                        if not label_dir.is_dir(): continue
                        candidate = label_dir / basename
                        if candidate.exists():
                            found_path = candidate
                            split = split_dir.name
                            label_name = label_dir.name
                            break
                    if found_path: break
                adv_candidate = found_path

        if adv_candidate is None:
            missing += 1
            adv_candidate = ADV_ROOT / attack_name / split / label_name / (Path(basename).stem + ".png")
            exists_flag = False
        else:
            found += 1
            exists_flag = True

        if MAKE_RELATIVE:
            relp = Path(REL_PREFIX) / attack_name / split / label_name / adv_candidate.name
            image_id_out = str(relp.as_posix()).replace("/", "\\") 
        else:
            image_id_out = str(adv_candidate.resolve())

        out_row = dict(row.drop(labels=["image_id_norm", "image_id_norm_lc", "basename"]))
        out_row["image_id"] = image_id_out
        rows.append(out_row)

    df_out = pd.DataFrame(rows)
    out_csv = OUT_DIR / f"Annotations_{attack_name}.csv"
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")
    summary[attack_name] = {"found": found, "missing": missing, "out_csv": out_csv}
    print(f" -> attack {attack_name}: found {found}, missing {missing}. Wrote: {out_csv}")

print("\nAll done. Summary:")
for k,v in summary.items():
    print(f"  {k}: found={v['found']}, missing={v['missing']}, csv={v['out_csv']}")

Loaded CSV: 19526 rows. Example: [{'image_id': 'E:\\Fruit_Quality_Split\\train\\bad\\IMG_2510.JPG', 'split': 'train', 'label': 1}, {'image_id': 'E:\\Fruit_Quality_Split\\train\\good\\IMG20200728182302_27596.jpg', 'split': 'train', 'label': 0}]
Attacks discovered: ['ANN_fgsm', 'ANN_pgd', 'CNN_fgsm', 'CNN_pgd', 'RNN_fgsm', 'RNN_pgd']

Processing attack: ANN_fgsm
 -> attack ANN_fgsm: found 19526, missing 0. Wrote: C:\Users\evri\Desktop\Διπλωματικη\Adversarial AI Attack Detection\Adversarial-AI-Attack-Detection\fruit_quality_adv_annotations\Annotations_ANN_fgsm.csv

Processing attack: ANN_pgd
 -> attack ANN_pgd: found 19526, missing 0. Wrote: C:\Users\evri\Desktop\Διπλωματικη\Adversarial AI Attack Detection\Adversarial-AI-Attack-Detection\fruit_quality_adv_annotations\Annotations_ANN_pgd.csv

Processing attack: CNN_fgsm
 -> attack CNN_fgsm: found 19526, missing 0. Wrote: C:\Users\evri\Desktop\Διπλωματικη\Adversarial AI Attack Detection\Adversarial-AI-Attack-Detection\fruit_quality_adv_anno